# SQL 분석

## 분석 목적
이벤트 로그와 Amazon 데이터를 SQL로 분석해
사용자 행동 패턴과 콘텐츠 품질의 관계 확인

## 분석 쿼리 목록
1. 세그먼트별 평균 completion_rate
2. 콘텐츠 품질 구간별 skip_rate
3. 세그먼트별 D7 Retention
4. 카테고리별 평균 품질 점수 및 completion_rate

In [2]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib
import platform

if platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

# DB 연결
conn = sqlite3.connect('amazon.db')

# events, users csv를 DB에 테이블로 저장
df_events = pd.read_csv('events.csv')
df_users = pd.read_csv('users.csv')

df_events.to_sql('events', conn, if_exists='replace', index=False)
df_users.to_sql('users', conn, if_exists='replace', index=False)

print("DB 연결 및 테이블 로드 완료")
print("테이블 목록:")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

DB 연결 및 테이블 로드 완료
테이블 목록:
       name
0  products
1    events
2     users


## Query 1. 세그먼트별 평균 completion_rate 및 skip_rate

In [3]:
query1 = """
SELECT 
    segment,
    ROUND(AVG(completion_rate), 3) AS avg_completion_rate,
    ROUND(AVG(CASE WHEN is_skipped = 1 THEN 1.0 ELSE 0.0 END), 3) AS skip_rate,
    COUNT(DISTINCT user_id) AS user_count
FROM events
GROUP BY segment
ORDER BY avg_completion_rate DESC
"""

result1 = pd.read_sql(query1, conn)
print(result1)

   segment  avg_completion_rate  skip_rate  user_count
0    power                0.796      0.105         108
1   casual                0.550      0.304         353
2  at_risk                0.347      0.549         251
3  churned                0.202      0.754         288


## Query 1 인사이트
- power user: completion 79.6%, skip 10.5% → 핵심 유지 대상
- churned user: completion 20.2%, skip 75.4% → 관심 기반 재유입 캠페인 필요
- at_risk user: completion 34.7%, skip 54.9% → 이탈 전 고품질 콘텐츠 노출로 방어

※ skip_rate는 합성 데이터에서 낮은 관심도를 나타내는 proxy 지표로 설계됨
   실제 서비스에서는 콘텐츠 스킵 이벤트 로그로 대체 가능

## Query 2. 콘텐츠 품질 구간별 skip_rate 및 completion_rate

In [4]:
query2 = """
SELECT 
    CASE 
        WHEN content_quality_score < 0.4 THEN '저품질 (0~0.4)'
        WHEN content_quality_score < 0.6 THEN '중품질 (0.4~0.6)'
        WHEN content_quality_score < 0.8 THEN '고품질 (0.6~0.8)'
        ELSE '최고품질 (0.8~1.0)'
    END AS quality_band,
    ROUND(AVG(completion_rate), 3) AS avg_completion_rate,
    ROUND(AVG(CASE WHEN is_skipped = 1 THEN 1.0 ELSE 0.0 END), 3) AS skip_rate,
    COUNT(*) AS event_count
FROM events
GROUP BY quality_band
ORDER BY avg_completion_rate DESC
"""

result2 = pd.read_sql(query2, conn)
print(result2)

     quality_band  avg_completion_rate  skip_rate  event_count
0  최고품질 (0.8~1.0)                0.714      0.267           30
1   고품질 (0.6~0.8)                0.673      0.250         1269
2   중품질 (0.4~0.6)                0.598      0.289        21200
3     저품질 (0~0.4)                0.506      0.335         2603


## Query 2 인사이트
- 품질이 낮을수록 completion_rate 낮고 skip_rate 높음
- 저품질 콘텐츠의 skip_rate = 33.5% vs 고품질 = 25.0%
- 서비스 액션: 저품질 콘텐츠 피드 노출 비율 제한, 고품질 우선 노출

## Query 3. 세그먼트별 D7 Retention

In [5]:
query3 = """
WITH first_visit AS (
    SELECT user_id, MIN(event_date) AS first_date
    FROM events
    GROUP BY user_id
),
revisit AS (
    SELECT e.user_id
    FROM events e
    JOIN first_visit f ON e.user_id = f.user_id
    WHERE JULIANDAY(e.event_date) - JULIANDAY(f.first_date) BETWEEN 1 AND 7
)
SELECT 
    u.segment,
    COUNT(DISTINCT u.user_id) AS total_users,
    COUNT(DISTINCT r.user_id) AS retained_users,
    ROUND(COUNT(DISTINCT r.user_id) * 100.0 / COUNT(DISTINCT u.user_id), 1) AS d7_retention_rate
FROM users u
LEFT JOIN revisit r ON u.user_id = r.user_id
GROUP BY u.segment
ORDER BY d7_retention_rate DESC
"""

result3 = pd.read_sql(query3, conn)
print(result3)

   segment  total_users  retained_users  d7_retention_rate
0    power          108              81               75.0
1   casual          353             168               47.6
2  at_risk          251              52               20.7
3  churned          288               0                0.0


## Query 3 인사이트
- Python 코호트 분석 결과와 SQL 결과 일치 → 데이터 정합성 검증
- power user D7 retention 75% → 핵심 유지 대상
- churned user D7 retention 0% → 재유입 캠페인 필요
- at_risk user 20.7% → 이탈 방어 전략 시급

## Query 4. 카테고리별 평균 품질 점수 및 completion_rate

In [7]:
query4 = """
SELECT 
    p.category_main,
    ROUND(AVG(e.content_quality_score), 3) AS avg_quality_score,
    ROUND(AVG(e.completion_rate), 3) AS avg_completion_rate,
    ROUND(AVG(CASE WHEN e.is_skipped = 1 THEN 1.0 ELSE 0.0 END), 3) AS skip_rate,
    COUNT(DISTINCT e.user_id) AS user_count
FROM events e
JOIN products p ON e.content_id = p.product_id
GROUP BY p.category_main
ORDER BY avg_quality_score DESC
"""

result4 = pd.read_sql(query4, conn)
print(result4)

           category_main  avg_quality_score  avg_completion_rate  skip_rate  \
0             Toys&Games              0.548                0.598      0.190   
1         OfficeProducts              0.544                0.616      0.282   
2        HomeImprovement              0.518                0.645      0.372   
3  Computers&Accessories              0.514                0.605      0.280   
4            Electronics              0.500                0.593      0.294   
5           Home&Kitchen              0.484                0.582      0.299   
6     MusicalInstruments              0.471                0.535      0.333   
7    Health&PersonalCare              0.469                0.428      0.583   
8          Car&Motorbike              0.421                0.466      0.458   

   user_count  
0          20  
1         386  
2          41  
3         924  
4         958  
5         970  
6          41  
7          12  
8          24  


## Query 4 인사이트
- Toys&Games, OfficeProducts → 품질 높고 completion 높음 → 추천 비율 확대
- Health&PersonalCare, Car&Motorbike → 품질 낮고 skip 높음 → 노출 비율 축소
- 서비스 액션:
  1. 저품질 카테고리 연속 노출 제한
  2. 저품질 카테고리 이후 다른 카테고리 콘텐츠 노출
  3. 카테고리별 품질 모니터링 대시보드 구축